# Use cases

Two concrete lookups on top of the mock data:

1. **per compound** — all targets it has been tested against
2. **per target** — all compounds tested against it

This connects to `probe.db`, the on-disk database built from `staging/_template`
by `examples/build_mock_db.py`. Run that script first if the file does not exist
yet:

```bash
uv run python examples/build_mock_db.py
```

In [1]:
from pathlib import Path

import pandas as pd

from probedb import ProbeDB

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 170)

DB_PATH = Path("..") / "probe.db"
assert DB_PATH.exists(), f"{DB_PATH} not found -- run examples/build_mock_db.py first"

db = ProbeDB(DB_PATH, create=False)

db.counts()

,table,rows
0,compound,3
1,chembl,3
2,uniprot,8
3,target,6
4,target_uniprot,9
5,bioactivity_source,10
6,bioactivity_group,8
7,bioactivity,13


## Use case 1: per compound

For every compound, answer four questions:

- **which targets** has it been measured against?
- **which set(s)** do the measurements come from (`source_db`: opnMe,
  Probes & Drugs, in-house, literature, ...)?
- **what is its main target** — the one with the strongest reported potency?
- **what is its selectivity** — how much weaker is the next best target on the
  same scale?

"Main target" and "selectivity" only compare rows that are actually
comparable: `bioactivity_type == "IC50"`, `unit == "nM"`, `relation == "="`.
Mixing pIC50 on -log(M) or a cell EC50 into the same ranking as a biochemical
IC50 would be comparing different things, so those stay out of the ranking.
Anything measured as a bound (`>`, `<`, ...) is a counter-screen, not a
potency, so it is reported separately and read qualitatively.

In [2]:
def compound_profile(db, compound):
    hits = db.bioactivities(compound=compound)

    targets = hits[["target_type", "target"]].drop_duplicates().reset_index(drop=True)
    sources = sorted(hits["source_db"].dropna().unique())

    comparable = hits[
        (hits.bioactivity_type == "IC50") & (hits.unit == "nM") & (hits.relation == "=")
    ]
    potency = (
        comparable.groupby(["target", "target_type"], as_index=False)["value"]
        .median()
        .sort_values("value")
        .reset_index(drop=True)
    )

    counter_screens = hits[hits.relation.isin([">", ">=", "<", "<="])]

    return targets, sources, potency, counter_screens


for compound in db.table("compound")["name"]:
    targets, sources, potency, counter_screens = compound_profile(db, compound)

    print(f"== {compound} ==")

    print(f"targets ({len(targets)}):")
    for _, row in targets.iterrows():
        print(f"  [{row.target_type}] {row.target}")

    print(f"derived from: {', '.join(sources) if sources else 'no source recorded'}")

    if potency.empty:
        print("main target: no comparable IC50 (nM, '=') data")
    else:
        best = potency.iloc[0]
        print(f"main target: {best.target}  (IC50 = {best.value:g} nM)")
        if len(potency) > 1:
            second = potency.iloc[1]
            fold = second.value / best.value
            print(
                f"selectivity: {fold:.1f}-fold vs {second.target} "
                f"(IC50 = {second.value:g} nM), the next best on the same scale"
            )
        else:
            print("selectivity: only one target with comparable IC50 (nM) data")

    if not counter_screens.empty:
        print("counter-screens (different scale, read qualitatively):")
        for _, row in counter_screens.iterrows():
            print(f"  {row.target}: {row.bioactivity_type} {row.relation} {row.value:g} {row.unit}")

    print()

== BI-2536 ==
targets (3):
  [protein] Serine/threonine-protein kinase PLK1
  [protein] Bromodomain-containing protein 4
  [complex] Cyclin-dependent kinase 1/cyclin B1
derived from: Probes & Drugs, in-house, literature, opnMe
main target: Serine/threonine-protein kinase PLK1  (IC50 = 0.965 nM)
selectivity: 1.2-fold vs Bromodomain-containing protein 4 (IC50 = 1.2 nM), the next best on the same scale
counter-screens (different scale, read qualitatively):
  Cyclin-dependent kinase 1/cyclin B1: IC50 > 10000 nM

== (+)-JQ1 ==
targets (2):
  [protein] Bromodomain-containing protein 4
  [protein] Bromodomain-containing protein 2
derived from: literature
main target: Bromodomain-containing protein 2  (IC50 = 130.75 nM)
selectivity: only one target with comparable IC50 (nM) data

== Olaparib ==
targets (3):
  [protein] Poly [ADP-ribose] polymerase 1
  [family] PARP 1, 2 and 3
  [protein] Serine/threonine-protein kinase PLK1
derived from: in-house, literature
main target: Poly [ADP-ribose] poly

## Use case 2: per target, all compounds

Same idea in the other direction: `db.bioactivities(target=...)` accepts a
target id, a UniProt accession or an HGNC symbol. Iterating `db.table("target")`
walks proteins, complexes and families alike.

In [3]:
def compounds_for_target(db, target_id):
    hits = db.bioactivities(target=target_id)
    return hits[["compound"]].drop_duplicates().reset_index(drop=True)


for _, target in db.table("target").iterrows():
    compounds = compounds_for_target(db, target.target_id)
    plural = "" if len(compounds) == 1 else "s"
    name = target["name"]
    print(f"{name} ({target.type}) -- {len(compounds)} compound{plural}")
    for _, row in compounds.iterrows():
        print(f"  {row.compound}")
    print()

Serine/threonine-protein kinase PLK1 (protein) -- 2 compounds
  BI-2536
  Olaparib

Bromodomain-containing protein 4 (protein) -- 2 compounds
  BI-2536
  (+)-JQ1

Bromodomain-containing protein 2 (protein) -- 1 compound
  (+)-JQ1

Poly [ADP-ribose] polymerase 1 (protein) -- 1 compound
  Olaparib

PARP 1, 2 and 3 (family) -- 1 compound
  Olaparib

Cyclin-dependent kinase 1/cyclin B1 (complex) -- 1 compound
  BI-2536

